In [1]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(model_id)

text = "The capital of France is [MASK]."
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

# To get predictions for the mask:
masked_index = inputs["input_ids"][0].tolist().index(tokenizer.mask_token_id)
predicted_token_id = outputs.logits[0, masked_index].argmax(axis=-1)
predicted_token = tokenizer.decode(predicted_token_id)
print("Predicted token:", predicted_token)
# Predicted token:  Paris


model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Predicted token:  Paris


In [3]:
### Get the embdding vector out

import torch
from transformers import AutoTokenizer, AutoModel

def get_bert_embedding(text, model_id="answerdotai/ModernBERT-base"):
    """
    Get BERT embedding for a given text.
    Returns the [CLS] token embedding as a numpy array.
    
    Args:
        text (str): Input text to embed
        model_id (str): HuggingFace model identifier
        
    Returns:
        numpy array: 768-dimensional embedding vector
    """
    # Initialize tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id)
    
    # Set model to evaluation mode
    model.eval()
    
    # Tokenize the text
    inputs = tokenizer(
        text,
        return_tensors="pt"
    )
    
    # Get BERT outputs
    with torch.no_grad():  # No need to calculate gradients
        outputs = model(**inputs)
        
    # Get the [CLS] token embedding (first token of last hidden state)
    embedding = outputs.last_hidden_state[:, 0, :]
    
    # Convert to numpy array and return
    return embedding.numpy().flatten()

# Example usage
if __name__ == "__main__":
    # Test texts
    texts = [
        "mangoes are the best",
        "i find bananas repulsive"
    ]
    
    # Get embeddings
    for text in texts:
        embedding = get_bert_embedding(text)
        print(f"\nText: {text}")
        print(f"Embedding shape: {embedding.shape}")
        print(f"First few dimensions: {embedding[:5]}")  # Show first 5 dimensions


Text: mangoes are the best
Embedding shape: (768,)
First few dimensions: [ 0.4189815  -0.81616974 -1.3294746  -0.32772404 -0.5284993 ]

Text: i find bananas repulsive
Embedding shape: (768,)
First few dimensions: [ 0.41122591 -0.49668095 -0.22513425  0.3575785  -0.46641928]


In [9]:
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
import numpy as np

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

class BERTClassifier(nn.Module):
    def __init__(self, bert_model, num_classes=2):
        super().__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_classes)  # 768 is BERT's hidden size

    def forward(self, input_ids, attention_mask):
        # Get BERT embeddings
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] token representation for classification
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

def train_model(model, train_loader, val_loader, num_epochs=3, learning_rate=2e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_accuracy = val_correct / val_total
        print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Val Accuracy: {val_accuracy:.4f}')

def predict(model, tokenizer, text):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    
    encoding = tokenizer(
        text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, predicted = torch.max(outputs, 1)
    
    return "like" if predicted.item() == 1 else "dislike"

# Example usage:
if __name__ == "__main__":
    # Initialize model and tokenizer
    model_id = "answerdotai/ModernBERT-base"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    bert_model = AutoModel.from_pretrained(model_id)
    
    # Sample training data
    texts = [
        "mangoes are the best",
        "i find bananas repulsive",
        "chocolate is amazing",
        "i hate spinach",
        # Add more examples...
    ]
    labels = [1, 0, 1, 0]  # 1 for like, 0 for dislike
    
    # Create dataset and dataloaders
    dataset = SentimentDataset(texts, labels, tokenizer)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8)
    
    # Initialize and train the model
    model = BERTClassifier(bert_model)
    train_model(model, train_loader, val_loader)
    
    # Test the model
    test_text = "mangoes are the best"
    prediction = predict(model, tokenizer, test_text)
    print(f"Text: {test_text}\nPrediction: {prediction}")
    test_text = "blueberries suck"
    prediction = predict(model, tokenizer, test_text)
    print(f"Text: {test_text}\nPrediction: {prediction}")


Epoch 1, Loss: 0.7659, Val Accuracy: 0.0000
Epoch 2, Loss: 0.7502, Val Accuracy: 0.0000
Epoch 3, Loss: 0.4756, Val Accuracy: 0.0000
Text: mangoes are the best
Prediction: dislike
Text: blueberries suck
Prediction: dislike


In [10]:
### Using a pretrained pipeline

import torch
from transformers import pipeline
from pprint import pprint

pipe = pipeline(
    "fill-mask",
    model="answerdotai/ModernBERT-base",
    torch_dtype=torch.bfloat16,
)

input_text = "He walked to the [MASK]."
results = pipe(input_text)
pprint(results)


Device set to use mps:0


[{'score': 0.115234375,
  'sequence': 'He walked to the door.',
  'token': 3369,
  'token_str': ' door'},
 {'score': 0.037353515625,
  'sequence': 'He walked to the office.',
  'token': 3906,
  'token_str': ' office'},
 {'score': 0.02734375,
  'sequence': 'He walked to the library.',
  'token': 6335,
  'token_str': ' library'},
 {'score': 0.02001953125,
  'sequence': 'He walked to the gate.',
  'token': 7394,
  'token_str': ' gate'},
 {'score': 0.02001953125,
  'sequence': 'He walked to the window.',
  'token': 3497,
  'token_str': ' window'}]
